In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define parameters globally to easily scale or change countries/tasks later
DATA_DIR = "Data" 
COUNTRIES = ['Belgium', 'Spain', 'Poland']
NRCA_TASKS = ['t_4A2a4', 't_4A2b2', 't_4A4a1']

# Import data from the O*NET database
task_data = pd.read_csv(os.path.join(DATA_DIR, "onet_tasks.csv"))

# Read employment data from Eurostat
# Passing sheet_name=None loads all sheets into a dictionary of DataFrames
eurostat_file = os.path.join(DATA_DIR, "Eurostat_employment_isco.xlsx")
isco_sheets = pd.read_excel(eurostat_file, sheet_name=None)

In [ ]:
# Combine all ISCO sheets dynamically and assign the ISCO category column
all_data_list = []
for sheet_name, df in isco_sheets.items():
    df['ISCO'] = int(sheet_name.replace('ISCO', ''))
    all_data_list.append(df)

all_data = pd.concat(all_data_list, ignore_index=True)

# Calculate worker totals and shares for each country
for country in COUNTRIES:
    # groupby().transform() computes the total for each TIME period and broadcasts it to all rows
    all_data[f'total_{country}'] = all_data.groupby('TIME')[country].transform('sum')
    
    # Calculate employment shares
    all_data[f'share_{country}'] = all_data[country] / all_data[f'total_{country}']

In [ ]:
# Extract 1-digit ISCO code 
task_data["isco08_1dig"] = task_data["isco08"].astype(str).str[:1].astype(int)

# Calculate the mean task values at a 1-digit level
aggdata = task_data.groupby("isco08_1dig").mean(numeric_only=True).drop(columns=["isco08"], errors='ignore')

# Combine the employment data with the aggregated task data
combined = pd.merge(all_data, aggdata, left_on='ISCO', right_on='isco08_1dig', how='left')

In [ ]:
def weighted_standardize(values, weights):
    """Calculates standardized values (mean 0, std 1) using a weighted average."""
    temp_mean = np.average(values, weights=weights)
    temp_var = np.average((values - temp_mean)**2, weights=weights)
    temp_sd = np.sqrt(temp_var)
    return (values - temp_mean) / temp_sd

# Loop through each country and each task to standardize the values
for country in COUNTRIES:
    for task in NRCA_TASKS:
        combined[f'std_{country}_{task}'] = weighted_standardize(
            combined[task], 
            combined[f'share_{country}']
        )

In [ ]:
# Calculate classic task content intensity and plot
plt.figure(figsize=(10, 6))

for country in COUNTRIES:
    # 1. Sum standardized tasks into a single NRCA score per country
    std_task_cols = [f'std_{country}_{task}' for task in NRCA_TASKS]
    combined[f'{country}_NRCA'] = combined[std_task_cols].sum(axis=1)

    # 2. Standardize the new NRCA aggregate
    combined[f'std_{country}_NRCA'] = weighted_standardize(
        combined[f'{country}_NRCA'], 
        combined[f'share_{country}']
    )

    # 3. Multiply by share to track changes over time
    combined[f'multip_{country}_NRCA'] = combined[f'std_{country}_NRCA'] * combined[f'share_{country}']

    # 4. Aggregate by TIME
    agg_country = combined.groupby("TIME")[f'multip_{country}_NRCA'].sum().reset_index()

    # Plot the line for the current country
    plt.plot(agg_country["TIME"], agg_country[f'multip_{country}_NRCA'], label=country, marker='o', markersize=4)

# Format the plot for better readability
plt.xticks(agg_country["TIME"][::3], rotation=45)
plt.title('Non-Routine Cognitive Analytical (NRCA) Task Intensity Over Time')
plt.xlabel('Time (Quarters)')
plt.ylabel('NRCA Intensity')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()